# analogOS · Demo Interativo
**Programmable Analogy Framework**

> *Analogy is not a way to explain systems. Analogy **is** the system.*

---

Este notebook demonstra o pipeline completo do **analogOS**:

```
scan → broadcast → candidate → propagate → compose
```

Domínio demonstrado: **analog-attention** (atenção semântica)

🔗 Repositório: [github.com/omega-Core-Dev/analogOS](https://github.com/omega-Core-Dev/analogOS)

## ⚙️ Setup — instalar dependências e clonar repositório

In [ ]:
!pip install numpy -q
!git clone https://github.com/omega-Core-Dev/analogOS.git 2>/dev/null || echo 'já clonado'
import sys, os
sys.path.insert(0, 'analogOS')
print('✅ Pronto')

## 🧱 Entity — unidade fundamental do espaço analógico

In [ ]:
from __future__ import annotations
import numpy as np
from dataclasses import dataclass, field
from typing import Any

@dataclass
class Entity:
    """
    Unidade fundamental do analogOS.
    id     : identificador único
    value  : dado associado (token, número, string...)
    vector : representação vetorial no espaço analógico
    tags   : etiquetas semânticas — filtro antes do cálculo vetorial
    """
    id: str
    value: Any
    vector: np.ndarray = field(default_factory=lambda: np.zeros(4))
    tags: list[str] = field(default_factory=list)
    meta: dict = field(default_factory=dict)

    def distance_to(self, other: Entity) -> float:
        if self.vector.shape != other.vector.shape:
            return 0.0
        return float(np.linalg.norm(self.vector - other.vector))

    def similarity_to(self, other: Entity) -> float:
        a, b = self.vector, other.vector
        norm = np.linalg.norm(a) * np.linalg.norm(b)
        if norm == 0:
            return 0.0
        return float(np.dot(a, b) / norm)

    def has_tag(self, tag: str) -> bool:
        return tag in self.tags

    def __repr__(self):
        return f"Entity(id={self.id!r}, value={self.value!r}, tags={self.tags})"

print('✅ Entity definida')

## 🔍 As 5 Primitivas Universais

In [ ]:
# ── SCAN ─────────────────────────────────────────────────────────────────────
# Percorre o espaço de entidades e constrói mapa de referência. O(n) — pago uma vez.

from typing import Callable

def scan(entities: list[Entity], key_fn: Callable[[Entity], str] | None = None) -> dict:
    if key_fn is None:
        key_fn = lambda e: e.id
    return {key_fn(e): e for e in entities}

print('✅ scan()')

In [ ]:
# ── BROADCAST ────────────────────────────────────────────────────────────────
# Fonte emite sinal; intensidade decai com distância. O(k).

def broadcast(source: Entity, ref_map: dict, intensity: float = 1.0,
              falloff: str = 'sqrt', noise: float = 0.0) -> dict[str, float]:
    import random
    signals = {}
    distances = {eid: source.distance_to(e) for eid, e in ref_map.items()}
    d_max = max(distances.values()) or 1.0
    for eid, dist in distances.items():
        d_norm = dist / d_max
        if falloff == 'quadratic':
            strength = intensity / (dist ** 2 + 1)
        elif falloff == 'sqrt':
            strength = intensity * (1 - (d_norm ** 0.5))
        else:
            strength = intensity * (1 - d_norm)
        if noise > 0:
            strength += random.uniform(-noise, noise)
        signals[eid] = max(0.0, strength)
    return signals

print('✅ broadcast()')

In [ ]:
# ── CANDIDATE ────────────────────────────────────────────────────────────────
# Filtra entidades que receberam sinal suficiente. O(k).

def candidate(entities: list[Entity], signals: dict[str, float],
              threshold: float = 0.4) -> list[Entity]:
    eligible = [e for e in entities if signals.get(e.id, 0.0) >= threshold]
    return sorted(eligible, key=lambda e: signals[e.id], reverse=True)

print('✅ candidate()')

In [ ]:
# ── PROPAGATE ────────────────────────────────────────────────────────────────
# Candidatos repassam sinal para vizinhos. Clusters emergentes. O(k·r).

def propagate(candidates: list[Entity], all_entities: list[Entity],
              signals: dict[str, float], social_factor: float = 0.5,
              radius: float | None = None) -> dict[str, float]:
    propagated = dict(signals)
    candidate_ids = {e.id for e in candidates}
    for cand in candidates:
        cand_signal = signals.get(cand.id, 0.0)
        for target in all_entities:
            if target.id in candidate_ids:
                continue
            dist = cand.distance_to(target)
            if radius is not None and dist > radius:
                continue
            repasse = cand_signal * social_factor / (dist + 1)
            propagated[target.id] = propagated.get(target.id, 0.0) + repasse
    return propagated

print('✅ propagate()')

In [ ]:
# ── COMPOSE ──────────────────────────────────────────────────────────────────
# Agrega cluster em saída unificada. 4 modos. O(k).

def compose(cluster: list[Entity], signals: dict[str, float],
            mode: str = 'top_k', top_k: int = 3) -> dict:
    ranked = sorted(cluster, key=lambda e: signals.get(e.id, 0.0), reverse=True)
    if mode == 'weighted_sum':
        total = sum(signals.get(e.id, 0.0) for e in ranked) or 1.0
        result = sum(signals.get(e.id, 0.0) / total *
                     (e.value if isinstance(e.value, (int, float)) else 1.0)
                     for e in ranked)
    elif mode == 'vote':
        votes = {}
        for e in ranked:
            v = str(e.value)
            votes[v] = votes.get(v, 0.0) + signals.get(e.id, 0.0)
        result = max(votes, key=votes.get)
    elif mode == 'top_k':
        result = [(e, signals.get(e.id, 0.0)) for e in ranked[:top_k]]
    else:  # concat
        result = [e.value for e in ranked]
    return {'mode': mode, 'result': result,
            'ranked': [(e, signals.get(e.id, 0.0)) for e in ranked]}

print('✅ compose()')

## 🚀 Pipeline Completo — Domínio: Atenção Semântica

**Scenario:** the query *'what is learning?'* searches for the most relevant tokens in memory.

In [ ]:
np.random.seed(42)

# ── Espaço de entidades (memória) ────────────────────────────────────────────
tokens = [
    Entity('t1', 'aprendizado',  np.array([0.9, 0.8, 0.1, 0.2]), tags=['conceito','cognitivo']),
    Entity('t2', 'rede neural',  np.array([0.8, 0.7, 0.2, 0.3]), tags=['conceito','ML']),
    Entity('t3', 'gradiente',    np.array([0.7, 0.5, 0.3, 0.4]), tags=['matematica','ML']),
    Entity('t4', 'memória',      np.array([0.6, 0.9, 0.1, 0.1]), tags=['conceito','cognitivo']),
    Entity('t5', 'banana',       np.array([0.1, 0.1, 0.9, 0.8]), tags=['fruta','alimento']),
    Entity('t6', 'fotossíntese', np.array([0.0, 0.2, 0.8, 0.9]), tags=['biologia','planta']),
]

# ── Query ────────────────────────────────────────────────────────────────────
query = Entity('query', 'o que é aprendizado?',
               np.array([0.85, 0.75, 0.15, 0.25]), tags=['query'])

print(f"📡 Query: '{query.value}'")
print(f"   Vetor: {query.vector}")

In [ ]:
# ── PASSO 1: SCAN ─────────────────────────────────────────────────────────────
ref_map = scan(tokens)
print(f"🔍 SCAN — {len(ref_map)} entidades mapeadas: {list(ref_map.keys())}")

In [ ]:
# ── PASSO 2: BROADCAST ───────────────────────────────────────────────────────
signals_bc = broadcast(source=query, ref_map=ref_map, intensity=1.0, falloff='sqrt')

print("📡 BROADCAST — sinais emitidos:\n")
for eid, sig in sorted(signals_bc.items(), key=lambda x: -x[1]):
    bar = '█' * int(sig * 30)
    print(f"  {eid} ({ref_map[eid].value:15s}) {sig:.3f}  {bar}")

In [ ]:
# ── PASSO 3: CANDIDATE ───────────────────────────────────────────────────────
THRESHOLD = 0.4
candidates = candidate(entities=tokens, signals=signals_bc, threshold=THRESHOLD)

print(f"✅ CANDIDATE (threshold={THRESHOLD}) — {len(candidates)} elegíveis:\n")
for e in candidates:
    print(f"  ✔ '{e.value}' | sinal={signals_bc[e.id]:.3f} | tags={e.tags}")

In [ ]:
# ── PASSO 4: PROPAGATE ───────────────────────────────────────────────────────
signals_prop = propagate(candidates=candidates, all_entities=tokens,
                         signals=signals_bc, social_factor=0.5)

print("🔁 PROPAGATE — sinais após propagação social:\n")
for eid, sig in sorted(signals_prop.items(), key=lambda x: -x[1]):
    delta = sig - signals_bc.get(eid, 0.0)
    delta_str = f" (+{delta:.3f} propagado)" if delta > 0.001 else ""
    print(f"  {eid} ({ref_map[eid].value:15s}) {sig:.3f}{delta_str}")

In [ ]:
# ── CLUSTER ──────────────────────────────────────────────────────────────────
half_thresh = THRESHOLD / 2
candidate_ids = {e.id for e in candidates}
propagated_extras = [
    e for e in tokens
    if e.id not in candidate_ids and signals_prop.get(e.id, 0.0) >= half_thresh
]
cluster = candidates + propagated_extras
print(f"🧩 Cluster: {[e.value for e in cluster]}")

In [ ]:
# ── PASSO 5: COMPOSE ─────────────────────────────────────────────────────────
output = compose(cluster=cluster, signals=signals_prop, mode='top_k', top_k=3)

print("🎯 COMPOSE (top_k=3) — resultado final:\n")
for i, (entity, sig) in enumerate(output['result'], 1):
    print(f"  {i}. '{entity.value}' — sinal={sig:.3f} | tags={entity.tags}")

print(f"\n{'='*50}")
print(f"Query   : '{query.value}'")
print(f"Resposta: {[e.value for e, _ in output['result']]}")
print(f"{'='*50}")

## 🔬 Experimente você mesmo

Mude os vetores, tags, threshold ou `social_factor` e observe como o pipeline responde.

Cada parâmetro tem semântica direta:
- **`intensity`** — força do sinal da query
- **`falloff`** — como o sinal decai com distância (`sqrt`, `linear`, `quadratic`)
- **`threshold`** — quão seletivo é o filtro de candidatos
- **`social_factor`** — quanto os candidatos repassam para vizinhos
- **`mode`** — como a saída é composta (`top_k`, `vote`, `weighted_sum`, `concat`)

In [ ]:
# ── Sua vez — modifique e explore ────────────────────────────────────────────

minha_query = Entity(
    id='minha_query',
    value='sua pergunta aqui',
    vector=np.array([0.5, 0.5, 0.5, 0.5]),  # mude este vetor
    tags=['query']
)

ref = scan(tokens)
sbc = broadcast(minha_query, ref, intensity=1.0, falloff='sqrt')
cands = candidate(tokens, sbc, threshold=0.3)  # threshold menor = mais candidatos
sprop = propagate(cands, tokens, sbc, social_factor=0.7)
cluster2 = cands + [e for e in tokens
                    if e.id not in {x.id for x in cands}
                    and sprop.get(e.id, 0) >= 0.15]
out2 = compose(cluster2, sprop, mode='top_k', top_k=3)

print(f"Resultado: {[e.value for e, _ in out2['result']]}")

---
**analogOS** · v0.1.0 · GPL-3.0  
Autor: Zaqueu Ribeiro · [github.com/omega-Core-Dev/analogOS](https://github.com/omega-Core-Dev/analogOS)

> *"O padrão estava sempre lá. Só precisava de um nome."*